In [1]:
import torch
from pathlib import Path
from torchvision import datasets
from torch.utils.data import random_split

# Use the existing dataset directly
DATASET_DIR = Path("../dataset/suitability")

dataset = datasets.ImageFolder(DATASET_DIR)

print("Total images:", len(dataset))
print("Classes:", dataset.class_to_idx)

# 70% train, 15% validation, 15% test
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

print()
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Total images: 21702
Classes: {'suitable': 0, 'unsuitable': 1}

Train: 15191
Validation: 3255
Test: 3256


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# All labels from ImageFolder
targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

# 70% train, 30% temporary
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=targets,
    random_state=42
)

# Split remaining 30% into 15% validation and 15% test
temp_targets = targets[temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_targets,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

print("Total:", len(dataset))
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

def count_classes(indices):
    values = targets[indices]
    return {
        "suitable": int(np.sum(values == 0)),
        "unsuitable": int(np.sum(values == 1))
    }

print("\nClass distribution:")
print("Train:", count_classes(train_idx))
print("Validation:", count_classes(val_idx))
print("Test:", count_classes(test_idx))

Total: 21702
Train: 15191
Validation: 3255
Test: 3256

Class distribution:
Train: {'suitable': 13931, 'unsuitable': 1260}
Validation: {'suitable': 2985, 'unsuitable': 270}
Test: {'suitable': 2986, 'unsuitable': 270}


In [3]:
from torchvision import transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Create separate ImageFolder datasets so each split can use
# the appropriate transforms without copying any images.
full_dataset_train = datasets.ImageFolder(DATASET_DIR, transform=train_transform)
full_dataset_eval = datasets.ImageFolder(DATASET_DIR, transform=val_test_transform)

train_dataset = Subset(full_dataset_train, train_idx)
val_dataset = Subset(full_dataset_eval, val_idx)
test_dataset = Subset(full_dataset_eval, test_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders ready!")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders ready!
Train batches: 950
Validation batches: 204
Test batches: 204


In [4]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# EfficientNet-B0 pretrained on ImageNet
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Replace final layer for binary classification
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    1
)

model = model.to(device)

# Handle class imbalance
# class 0 = suitable
# class 1 = unsuitable
# Calculate class weight from the current training split
train_targets = targets[train_idx]

num_suitable = int(np.sum(train_targets == 0))
num_unsuitable = int(np.sum(train_targets == 1))

pos_weight = torch.tensor(
    [num_suitable / num_unsuitable],
    device=device,
    dtype=torch.float32
)

print("Training class counts:")
print("Suitable:", num_suitable)
print("Unsuitable:", num_unsuitable)
print("Positive class weight:", pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Model:", "EfficientNet-B0")
print("Positive-class weight:", pos_weight.item())
print("Model ready!")

Device: cuda
Training class counts:
Suitable: 13931
Unsuitable: 1260
Positive class weight: 11.05634880065918
Model: EfficientNet-B0
Positive-class weight: 11.05634880065918
Model ready!


In [5]:
from tqdm.auto import tqdm

NUM_EPOCHS = 5

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Train")

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= train_total
    train_acc = train_correct / train_total

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Val")

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc:  {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": "efficientnet_b0",
                "class_names": ["suitable", "unsuitable"],
                "threshold": 0.5,
                "image_size": 224
            },
            "../models/lesion_suitability_efficientnet_b0_realworld.pth"
        )

        print("✓ Best model saved")

print()
print("Training complete!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

Epoch 1/5 - Train:   0%|          | 0/950 [00:00<?, ?it/s]

Epoch 1/5 - Val:   0%|          | 0/204 [00:00<?, ?it/s]


Epoch 1/5
Train Loss: 0.4313
Train Acc:  0.8899
Val Loss:   0.3205
Val Acc:    0.9124
✓ Best model saved


Epoch 2/5 - Train:   0%|          | 0/950 [00:00<?, ?it/s]

Epoch 2/5 - Val:   0%|          | 0/204 [00:00<?, ?it/s]

...

Epoch 4/5 - Val:   0%|          | 0/204 [00:00<?, ?it/s]


Epoch 4/5
Train Loss: 0.3012
Train Acc:  0.9138
Val Loss:   0.3017
Val Acc:    0.9146
✓ Best model saved


Epoch 5/5 - Train:   0%|          | 0/950 [00:00<?, ?it/s]

Epoch 5/5 - Val:   0%|          | 0/204 [00:00<?, ?it/s]


Epoch 5/5
Train Loss: 0.3089
Train Acc:  0.9109
Val Loss:   0.3037
Val Acc:    0.9155
✓ Best model saved

Training complete!
Best validation accuracy: 0.9155


In [6]:
import torch
from tqdm.auto import tqdm

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Testing:   0%|          | 0/204 [00:00<?, ?it/s]

Test Accuracy: 0.9082
Test Accuracy: 90.82%


In [7]:
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms

# Path to the held-out ISIC test images
ISIC_TEST_DIR = Path("/mnt/c/Users/dhruv/isic_melanoma_test")

# Load the NEW real-world suitability model
checkpoint = torch.load(
    "../models/lesion_suitability_efficientnet_b0_realworld.pth",
    map_location=device
)

from torchvision.models import efficientnet_b0

isic_model = efficientnet_b0(weights=None)
isic_model.classifier[1] = torch.nn.Linear(
    isic_model.classifier[1].in_features,
    1
)

isic_model.load_state_dict(checkpoint["model_state_dict"])
isic_model = isic_model.to(device)
isic_model.eval()

# Same preprocessing used during validation/testing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

suitable_count = 0
unsuitable_count = 0
failed_count = 0

probabilities = []

for image_path in sorted(ISIC_TEST_DIR.glob("*.jpg")):
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = isic_model(image_tensor)
            probability = torch.sigmoid(output).item()

        probabilities.append(probability)

        if probability < 0.5:
            suitable_count += 1
        else:
            unsuitable_count += 1

    except Exception:
        failed_count += 1

print("ISIC test images:", len(probabilities))
print("Predicted suitable:", suitable_count)
print("Predicted unsuitable:", unsuitable_count)
print("Failed:", failed_count)

if probabilities:
    print("Suitable prediction rate:", suitable_count / len(probabilities) * 100)
    print("Unsuitable prediction rate:", unsuitable_count / len(probabilities) * 100)

ISIC test images: 1921
Predicted suitable: 1911
Predicted unsuitable: 10
Failed: 0
Suitable prediction rate: 99.47943779281624
Unsuitable prediction rate: 0.5205622071837585


In [8]:
from pathlib import Path
from PIL import Image
import torch

UNSUITABLE_DIR = Path("../dataset/suitability/unsuitable")

suitable_count = 0
unsuitable_count = 0
failed_count = 0

for image_path in sorted(UNSUITABLE_DIR.rglob("*")):
    if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
        continue

    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = isic_model(image_tensor)
            probability = torch.sigmoid(output).item()

        # 0 = suitable, 1 = unsuitable
        if probability < 0.5:
            suitable_count += 1
        else:
            unsuitable_count += 1

    except Exception:
        failed_count += 1

total = suitable_count + unsuitable_count

print("Real-world unsuitable images:", total)
print("Predicted suitable:", suitable_count)
print("Predicted unsuitable:", unsuitable_count)
print("Failed:", failed_count)

if total:
    print("Correct rejection rate:",
          unsuitable_count / total * 100)
    print("False suitable rate:",
          suitable_count / total * 100)

Real-world unsuitable images: 1800
Predicted suitable: 1
Predicted unsuitable: 1799
Failed: 0
Correct rejection rate: 99.94444444444444
False suitable rate: 0.05555555555555555
